In [1]:
import pandas as pd
import re
from tqdm import tqdm
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('stopwords')
nltk.download('punkt_tab')
tqdm.pandas()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
df = pd.read_csv("rawData/dataX.csv")
df = df[['created_at', 'user_id_str', 'full_text']]
df['full_text'] = df['full_text'].fillna('')
df.drop_duplicates(subset=['full_text'], inplace=True)
print(f"Data loaded: {len(df)} rows")

Data loaded: 1882 rows


In [3]:
def convert_emoji(text):
    if isinstance(text, str):
        return emoji.demojize(text, delimiters=(" ", " "))
    return text

In [4]:
def clean_text(text):
    text = convert_emoji(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

In [ ]:
df['clean_text'] = df['full_text'].progress_apply(clean_text)
print("Cleaning selesai")

100%|██████████| 1882/1882 [00:00<00:00, 8740.26it/s] 

Cleaning selesai


In [ ]:
kamus = pd.read_excel("kamuskatabaku.xlsx")
kamus_dict = dict(zip(kamus['tidak_baku'].str.lower(), kamus['kata_baku']))

def normalize_text(text):
    words = text.split()
    return " ".join([kamus_dict.get(w, w) for w in words])

df['normalized'] = df['clean_text'].progress_apply(normalize_text)
print(f"Normalisasi selesai. Kamus: {len(kamus_dict)} entri")

100%|██████████| 1882/1882 [00:00<00:00, 268849.16it/s]

Normalisasi selesai. Kamus: 4347 entri


In [7]:
df['tokens'] = df['normalized'].progress_apply(
    lambda x: word_tokenize(x) if isinstance(x, str) else []
)
print("Tokenization selesai")

100%|██████████| 1882/1882 [00:00<00:00, 11902.31it/s]

Tokenization selesai


In [ ]:
stop_words = set(stopwords.words('indonesian'))

def remove_stopwords(tokens):
    return [w for w in tokens if w not in stop_words and len(w)>2]

df['tokens'] = df['tokens'].progress_apply(remove_stopwords)
print("Stopword removal selesai")

100%|██████████| 1882/1882 [00:00<00:00, 268785.08it/s]

Stopword removal selesai


In [9]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens):
    return [stemmer.stem(w) for w in tokens]

df['stemmed'] = df['tokens'].progress_apply(stem_tokens)
print("Stemming selesai")

100%|██████████| 1882/1882 [02:31<00:00, 12.46it/s]

Stemming selesai


In [ ]:
print("\n" + "="*60)
print("VERIFIKASI HASIL PREPROCESSING")
print("="*60)

print(f"\nKolom di DataFrame: {df.columns.tolist()}")

print("\nContoh hasil (baris pertama):")
print(f"Stemmed:   {df['stemmed'].iloc[0][:10]}...")


VERIFIKASI HASIL PREPROCESSING

Kolom di DataFrame: ['created_at', 'user_id_str', 'full_text', 'clean_text', 'normalized', 'tokens', 'stemmed', 'final_text']

Contoh hasil (baris pertama):
Stemmed:   ['performa', 'mesin', 'maksimal', 'boros', 'bensin', 'ajar', 'pandu', 'campur', 'etanol', 'amp']...


In [12]:
df.to_csv("cleanData/twitter_X_preprocessed.csv", index=False)
print("\nFile berhasil disimpan!")


File berhasil disimpan!
